In [1]:
import pandas as pd
import numpy as np

# Step 1: Load dataset and standardize to English for GitHub portfolio
file_path = 'dataset_hotel_modelo_python_panama.csv'
df = pd.read_csv(file_path)

# Rename columns to English standard (snake_case)
df = df.rename(columns={
    'ID_Reserva': 'booking_id',
    'Hotel': 'hotel_name',
    'Mercado_Origen': 'source_market',
    'Fecha_Reserva': 'booking_date',
    'Fecha_CheckIn': 'checkin_date',
    'Booking_Window': 'booking_window_days',
    'Revenue_USD': 'revenue_usd'
})

# Standardize market labels to English
market_mapping = {
    'Panamá (Local)': 'Panama (Local)',
    'Colombia': 'Colombia',
    'USA': 'USA'
}
df['source_market'] = df['source_market'].map(market_mapping)

# Convert date strings to datetime objects for time-series manipulation
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['checkin_date'] = pd.to_datetime(df['checkin_date'])

# Export clean English dataset for the repository
df.to_csv('hotel_booking_demand_panama_en.csv', index=False)

# Exploratory Data Analysis (EDA): Key statistical metrics by source market
eda_summary = df.groupby('source_market').agg(
    total_bookings=('booking_id', 'count'),
    total_revenue_usd=('revenue_usd', 'sum'),
    mean_adr_usd=('revenue_usd', 'mean'),
    median_adr_usd=('revenue_usd', 'median'),
    std_adr_usd=('revenue_usd', 'std'),
    mean_lead_time=('booking_window_days', 'mean'),
    median_lead_time=('booking_window_days', 'median'),
    min_lead_time=('booking_window_days', 'min'),
    max_lead_time=('booking_window_days', 'max')
).reset_index()

# Calculate revenue share percentage
eda_summary['revenue_share_pct'] = (
    eda_summary['total_revenue_usd'] / eda_summary['total_revenue_usd'].sum()
) * 100

print(eda_summary.round(2))

    source_market  total_bookings  total_revenue_usd  mean_adr_usd  \
0        Colombia             208          249599.92        1200.0   
1  Panama (Local)            1444          649799.88         450.0   
2             USA             100           99999.97        1000.0   

   median_adr_usd  std_adr_usd  mean_lead_time  median_lead_time  \
0         1195.16       106.05           41.79              42.0   
1          449.55        44.18           10.06              10.0   
2          984.80       106.96           42.38              43.0   

   min_lead_time  max_lead_time  revenue_share_pct  
0             25             60              24.97  
1              1             19              65.02  
2             25             60              10.01  


In [2]:
# Convert date columns back to datetime objects
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['checkin_date'] = pd.to_datetime(df['checkin_date'])

# 1. Data Integrity Check (Poka-Yoke): Recalculate booking window to ensure accuracy
df['booking_window_days'] = (df['checkin_date'] - df['booking_date']).dt.days

# 2. Lead-Time Bucketing using pd.cut()
# Define tactical bins and labels for Paid Media campaign structuring
bins = [0, 7, 14, 24, 35, 45, 90]
labels = [
    '01-07 Days (Last Minute)',
    '08-14 Days (Short Term)',
    '15-24 Days (Mid-Short)',
    '25-35 Days (1 Month Out)',
    '36-45 Days (1.5 Months Out)',
    '46-60 Days (2 Months Out)'
]

df['lead_time_bucket'] = pd.cut(
    df['booking_window_days'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

# 3. Seasonality & Temporal Feature Extraction
# Extract month numbers and short names (Jan, Feb, etc.)
df['booking_month_num'] = df['booking_date'].dt.month
df['booking_month_name'] = df['booking_date'].dt.strftime('%b')

df['checkin_month_num'] = df['checkin_date'].dt.month
df['checkin_month_name'] = df['checkin_date'].dt.strftime('%b')

# Extract day of the week for booking behavior analysis
df['booking_day_of_week'] = df['booking_date'].dt.day_name()

# Define chronological order for months so plots don't sort alphabetically
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df['booking_month_name'] = pd.Categorical(df['booking_month_name'], categories=month_order, ordered=True)
df['checkin_month_name'] = pd.Categorical(df['checkin_month_name'], categories=month_order, ordered=True)

# Save the feature-engineered dataset for Step 3
df.to_csv('hotel_booking_demand_featured_en.csv', index=False)

# 4. Generate Verification Pivot Tables (Revenue USD and Intra-Market Share %)
pivot_revenue = pd.pivot_table(
    df,
    values='revenue_usd',
    index='source_market',
    columns='lead_time_bucket',
    aggfunc='sum',
    fill_value=0,
    observed=False
)

# Calculate percentage distribution per row (axis=0 divides by row total)
pivot_share = pivot_revenue.div(pivot_revenue.sum(axis=1), axis=0) * 100

print(pivot_revenue.round(2))
print(pivot_share.round(2))

lead_time_bucket  01-07 Days (Last Minute)  08-14 Days (Short Term)  \
source_market                                                         
Colombia                              0.00                     0.00   
Panama (Local)                   232863.97                241239.25   
USA                                   0.00                     0.00   

lead_time_bucket  15-24 Days (Mid-Short)  25-35 Days (1 Month Out)  \
source_market                                                        
Colombia                            0.00                  84137.82   
Panama (Local)                 175696.66                      0.00   
USA                                 0.00                  32537.92   

lead_time_bucket  36-45 Days (1.5 Months Out)  46-60 Days (2 Months Out)  
source_market                                                             
Colombia                             66840.40                   98621.70  
Panama (Local)                           0.00                       

In [5]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Re-enforce categorical ordering after loading from CSV
lead_time_order = [
    '01-07 Days (Last Minute)',
    '08-14 Days (Short Term)',
    '15-24 Days (Mid-Short)',
    '25-35 Days (1 Month Out)',
    '36-45 Days (1.5 Months Out)',
    '46-60 Days (2 Months Out)'
]
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df['lead_time_bucket'] = pd.Categorical(df['lead_time_bucket'], categories=lead_time_order, ordered=True)
df['booking_month_name'] = pd.Categorical(df['booking_month_name'], categories=month_order, ordered=True)
df['checkin_month_name'] = pd.Categorical(df['checkin_month_name'], categories=month_order, ordered=True)

# Set global visual style for portfolio-grade charts
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

# ==========================================
# CHART 1: Market vs. Lead-Time Bucket Heatmap (Revenue & Share %)
# ==========================================
pivot_rev = pd.pivot_table(
    df, values='revenue_usd', index='source_market', columns='lead_time_bucket',
    aggfunc='sum', fill_value=0, observed=False
)
pivot_pct = pivot_rev.div(pivot_rev.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Panel A: Absolute Revenue (USD)
sns.heatmap(
    pivot_rev, ax=axes[0], cmap='YlGnBu', annot=True, fmt=',.0f',
    cbar_kws={'label': 'Revenue (USD)'}, linewidths=1, linecolor='white'
)
axes[0].set_title('A. Total Revenue (USD) by Source Market & Booking Window', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Source Market', fontweight='bold')

# Panel B: Intra-Market Revenue Share (%)
sns.heatmap(
    pivot_pct, ax=axes[1], cmap='OrRd', annot=True, fmt='.1f',
    cbar_kws={'label': 'Intra-Market Share (%)'}, linewidths=1, linecolor='white'
)
axes[1].set_title('B. Intra-Market Revenue Share (%) by Booking Window (Budget Weighting Guide)', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Lead-Time Bucket (Days Before Check-In)', fontweight='bold')
axes[1].set_ylabel('Source Market', fontweight='bold')
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.savefig('01_lead_time_heatmap_by_market.png', dpi=300, bbox_inches='tight')
plt.close()

# ==========================================
# CHART 2: Booking Month vs. Check-In Month Lag (When We Spend vs. When They Stay)
# ==========================================
booking_monthly = pd.pivot_table(
    df, values='revenue_usd', index='source_market', columns='booking_month_name',
    aggfunc='sum', fill_value=0, observed=False
)
checkin_monthly = pd.pivot_table(
    df, values='revenue_usd', index='source_market', columns='checkin_month_name',
    aggfunc='sum', fill_value=0, observed=False
)

# Normalize by row (%) so each market's monthly seasonality curve is clearly visible
booking_monthly_pct = booking_monthly.div(booking_monthly.sum(axis=1), axis=0) * 100
checkin_monthly_pct = checkin_monthly.div(checkin_monthly.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

sns.heatmap(
    booking_monthly_pct, ax=axes[0], cmap='Blues', annot=True, fmt='.1f',
    cbar_kws={'label': '% of Annual Market Revenue'}, linewidths=1, linecolor='white'
)
axes[0].set_title('A. DEMAND GENERATION (BOOKING MONTH): Share of Annual Revenue (%) by Month of Purchase [When to Push Paid Media]', fontsize=12, fontweight='bold', pad=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Source Market', fontweight='bold')

sns.heatmap(
    checkin_monthly_pct, ax=axes[1], cmap='Greens', annot=True, fmt='.1f',
    cbar_kws={'label': '% of Annual Market Revenue'}, linewidths=1, linecolor='white'
)
axes[1].set_title('B. HOTEL OCCUPANCY (CHECK-IN MONTH): Share of Annual Revenue (%) by Month of Arrival [When Guests Actually Stay]', fontsize=12, fontweight='bold', pad=12)
axes[1].set_xlabel('Calendar Month', fontweight='bold')
axes[1].set_ylabel('Source Market', fontweight='bold')

plt.tight_layout()
plt.savefig('02_seasonality_booking_vs_checkin_lag.png', dpi=300, bbox_inches='tight')
plt.close()

In [6]:
df['booking_month_name'] = pd.Categorical(df['booking_month_name'], categories=month_order, ordered=True)

# Define Target Annual Paid Media Budget ($100,000 USD for a 10x Target ROAS on ~$1M Revenue)
TARGET_ANNUAL_BUDGET_USD = 100000.0
TARGET_ROAS = df['revenue_usd'].sum() / TARGET_ANNUAL_BUDGET_USD

# 1. Calculate Historical Revenue by Market and Booking Month (When Demand Occurs)
monthly_demand_rev = pd.pivot_table(
    df,
    values='revenue_usd',
    index='booking_month_name',
    columns='source_market',
    aggfunc='sum',
    fill_value=0,
    observed=False
)

# 2. Calculate Budget Allocation Weight (%) based on Total Revenue Contribution
total_rev = df['revenue_usd'].sum()
budget_allocation_usd = (monthly_demand_rev / total_rev) * TARGET_ANNUAL_BUDGET_USD

# Add Total Monthly Budget and Monthly Share (%)
budget_allocation_usd['Total_Monthly_Budget_USD'] = budget_allocation_usd.sum(axis=1)
budget_allocation_usd['Monthly_Budget_Share_Pct'] = (
    budget_allocation_usd['Total_Monthly_Budget_USD'] / TARGET_ANNUAL_BUDGET_USD
) * 100

# 3. Create Actionable Campaign Strategy Table by Market (Lead-Time & CPA Guardrails)
market_strategy = df.groupby('source_market').agg(
    Historical_Revenue_USD=('revenue_usd', 'sum'),
    Avg_Ticket_ADR_USD=('revenue_usd', 'mean'),
    Mean_Booking_Window_Days=('booking_window_days', 'mean'),
    P25_Lead_Time_Days=('booking_window_days', lambda x: np.percentile(x, 25)),
    P75_Lead_Time_Days=('booking_window_days', lambda x: np.percentile(x, 75))
).reset_index()

market_strategy['Annual_Budget_USD'] = (market_strategy['Historical_Revenue_USD'] / total_rev) * TARGET_ANNUAL_BUDGET_USD
market_strategy['Target_Conversions_Bookings'] = (
    market_strategy['Historical_Revenue_USD'] / market_strategy['Avg_Ticket_ADR_USD']
).round(0)
market_strategy['Max_Target_CPA_USD'] = (
    market_strategy['Annual_Budget_USD'] / market_strategy['Target_Conversions_Bookings']
).round(2)

# Save deliverables to CSV for GitHub Repo
budget_allocation_usd.round(2).to_csv('paid_media_monthly_budget_pacing.csv')
market_strategy.round(2).to_csv('paid_media_market_kpis_targets.csv', index=False)

# 4. Generate Chart 3: Annual Paid Media Budget Pacing by Market & Month
fig, ax = plt.subplots(figsize=(13, 6))

markets = ['Panama (Local)', 'Colombia', 'USA']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bottom = np.zeros(len(month_order))
for market, color in zip(markets, colors):
    values = budget_allocation_usd[market].values
    ax.bar(month_order, values, bottom=bottom, label=market, color=color, width=0.65, edgecolor='white', alpha=0.9)
    bottom += values

# Annotate total monthly budget on top of each stacked bar
for i, total in enumerate(budget_allocation_usd['Total_Monthly_Budget_USD']):
    ax.text(i, total + 180, f"${total:,.0f}", ha='center', va='bottom', fontsize=9, fontweight='bold', color='#222222')

ax.set_title('Annual Paid Media Budget Pacing by Source Market ($100k Total Budget | 10x Target ROAS)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Execution Month (Booking Month - Demand Generation)', fontweight='bold')
ax.set_ylabel('Allocated Ad Spend (USD)', fontweight='bold')
ax.set_ylim(0, max(budget_allocation_usd['Total_Monthly_Budget_USD']) * 1.15)
ax.legend(title='Source Market', frameon=True, loc='upper right')

plt.tight_layout()
plt.savefig('03_paid_media_budget_pacing_chart.png', dpi=300, bbox_inches='tight')
plt.close()